In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


show me rock

In [5]:
from cgitb import reset


openai = OpenAI()

todos = []
completed = []

def get_todo_report():
    result = ""
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result
get_todo_report()

def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [6]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [7]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [8]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [9]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

todos, completed = [], []
loop(messages)

In [10]:
todos, completed = [], []
loop(messages)

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance).
Todo #3: Set up relative motion equation using departure times and speeds.
Todo #4: Solve for meeting time and report in clock time.

Let D be the BostonNew York distance along the track. Boston train leaves 2:00 pm at 60 mph. New York train leaves
3:00 pm at 80 mph toward Boston. They travel toward each other, so distances add to D at meeting time.

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance).
Todo #3: Set up relative motion equation using departure times and speeds.
Todo #4: Solve for meeting time and report in clock time.

Distance not provided. Use a reasonable estimate: BostonNew York is about 215 miles (common driving distance 
~215216 mi). Take D = 215 mi.

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance).
Todo #3: Set up relative motion equation using departure times and speeds.
Todo #4: Solve for meeting time and report in clock time.

Let t = hours after 2:00 pm when they meet. Then Boston train travels 60t miles. New York train starts at 3:00 pm, 
i.e., 1 hour later, so it travels 80(t-1) miles (for t\ge1). Equation: 60t + 80(t-1) = D.

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance).
Todo #3: Set up relative motion equation using departure times and speeds.
Todo #4: Solve for meeting time and report in clock time.

Solve: 60t + 80(t-1)=215 => 60t+80t-80=215 => 140t=295 => t=295/140=2.1071 h after 2:00 pm. That is 2 h + 0.1071 h 
= 2 h + 6.43 min ≈ 2:06 pm after 4:00? Wait: 2:00 pm + 2.1071 h = 4:06 pm (about 4:06:26 pm).

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance).
Todo #3: Set up relative motion equation using departure times and speeds.
Todo #4: Solve for meeting time and report in clock time.

Assumption (missing info): Boston–New York distance \u2248 215 miles.

Let t = hours after 2:00 pm when they meet.

- Boston train distance: 60t
- New York train leaves 1 hour later, so its travel time is (t-1) and distance is 80(t-1)

They meet when distances sum to 215:
60t + 80(t-1) = 215  
140t - 80 = 215  
140t = 295  
t = 295/140 \u2248 2.107 hours

So the meeting time is:
2:00 pm + 2.107 h \u2248 4:06 pm (about 4:06:26 pm).